In [75]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

model = "claude-haiku-4-5"

In [76]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [ ]:
import json  

def generate_dataset():

	prompt = """

Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts

that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,

each representing task that requires Python, JSON, or a Regex to complete.

  

Example output:

```json

[

	{

		"task": "Description of task",
		
	},
		
	...additional

]

``` 
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.



"""
	messages = []
	add_user_message(messages, prompt)
	add_assistant_message(messages, "```json")  # 프리필링
	text = chat(messages, stop_sequences=["```"])  # 정지 시퀀스
	return json.loads(text)

In [78]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [79]:
dataset

[{'task': 'Write a Python script to list all S3 buckets in an AWS account using boto3'},
 {'task': 'Create a JSON CloudFormation template to provision an EC2 instance with a security group'},
 {'task': 'Write a Regex pattern to validate AWS IAM role ARNs'},
 {'task': 'Write a Python script to upload a file to S3 and generate a presigned URL'},
 {'task': 'Create a JSON template for an AWS Lambda function environment variables configuration'},
 {'task': 'Write a Regex pattern to extract AWS account IDs from CloudFormation stack names'},
 {'task': 'Write a Python script to describe all EC2 instances and filter by tag'},
 {'task': 'Create a JSON policy document for an S3 bucket with specific access restrictions'},
 {'task': 'Write a Regex pattern to validate AWS S3 bucket names'},
 {'task': 'Write a Python script to create a DynamoDB table and add items'},
 {'task': 'Create a JSON configuration for AWS RDS database cluster parameters'},
 {'task': 'Write a Regex pattern to match AWS CloudWa

In [80]:
def grade_by_model(test_case, output):
    """LLM-as-Judge: 다른 Claude 호출로 출력을 평가"""
    eval_prompt = f"""

You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.  

Original Task:
<task>
{test_case["task"]}
</task> 

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10  

Respond with JSON. Keep your response concise and direct.

Example response shape:

{{
"strengths": string[],
"weaknesses": string[],
"reasoning": string,
"score": number
}}

"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```code")
    eval_text = chat(messages, stop_sequences=["```"])	
    return json.loads(eval_text)

In [81]:
# Functions to validate the output structure

import re
import ast  

def validate_json(text):
	try:
		json.loads(text.strip())
		return 10
	except json.JSONDecodeError:
		return 0
		
def validate_python(text):
	try:
		ast.parse(text.strip())
		return 10
	except SyntaxError:
		return 0 

def validate_regex(text):
	try:
		re.compile(text.strip())
		return 10
	except re.error:
		return 0

def grade_syntax(response, test_case):
	format = test_case["format"]
	if format == "json":
		return validate_json(response)
	elif format == "python":
		return validate_python(response)
	else:
		return validate_regex(response)

In [ ]:
def run_prompt(test_case):
    """프롬프트와 테스트 케이스를 병합하여 실행"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation


"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [83]:
def run_test_case(test_case):
    """run_prompt 호출 후 결과를 채점"""
    output = run_prompt(test_case)

    # TODO - 채점 로직 (다음 섹션에서 구현)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output, test_case)
    final_score = (model_score + syntax_score) / 2  # 단순 평균으로 최종 점수 계산
    
    return {
        "output": output,
        "test_case": test_case,
        "score": final_score,
        "reasoning": reasoning
    }

In [84]:
def run_eval(dataset):
    """데이터셋의 모든 테스트 케이스를 순차 실행"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average = sum(result["score"] for result in results) / len(results)
    print(f"Average Score: {average:.2f}")

    return results

In [85]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

KeyError: 'format'